# 第3回：AI Agent用の参照テーブルを作成する

既存の `Census_PoC_Cluster` に接続し、上から実行します。AI Computeでは実行しません。

ブログの検証環境では、AIDP上で2つのDeltaテーブル（61件・305件）の作成とSpark SQLの実行を確認しました。この公開版のコードセルは、その実行セットから変更していません。新しい環境では、入力パス・権限・カタログを確認して実行してください。

第2回の元データは読み取りのみで、新規の2テーブルを作成します。既定では同名テーブルの上書きを行いません。


## 1) 入力と保存先

標準はVolume内の第2回データです。PathはPython向けの `/Volumes/...` を指定します。URIの `file://` は付けません。

Volume読取ができない場合は `USE_BUNDLED_INPUTS=True` とし、同梱 `inputs` フォルダーをアップロードした実際のWorkspaceパスを `BUNDLED_INPUT_DIR` に指定します。

In [ ]:
from pathlib import Path
import json

USE_BUNDLED_INPUTS = False
BUNDLED_INPUT_DIR = Path("/Workspace/実際のアップロード先/inputs")
VOLUME_ROOT = Path("/Volumes/blog_catalog/opendata/census_work")
COMPARISON_RUN_ID = "compare61_20261020_01"
AI_RUN_ID = "change_ml_20261020_01"
CATALOG = "blog_catalog"
SCHEMA = "opendata"
TABLE_SUFFIX = "v1"
ALLOW_REPLACE_PART3_TABLES = False

CSV_DIR = VOLUME_ROOT / "comparisons" / COMPARISON_RUN_ID / "csv" / "tokyo_changes"
ML_ROOT = VOLUME_ROOT / "ai_runs" / AI_RUN_ID / "ml"
if USE_BUNDLED_INPUTS:
    CSV_PATHS = [BUNDLED_INPUT_DIR / "tokyo_changes.csv"]
    ML_ROOT = BUNDLED_INPUT_DIR
else:
    CSV_PATHS = sorted(CSV_DIR.glob("part-*.csv"))
print("CSV:", [str(p) for p in CSV_PATHS])
print("ML JSON:", str(ML_ROOT))
if not CSV_PATHS or any(not p.is_file() for p in CSV_PATHS):
    raise FileNotFoundError("CSVパスを確認してください。パスは推測で変更せず、第2回Notebookと照合します。")
for name in ["run_metadata.json", "model_parameters.json", "profile_payload.json", "comparison_validation_snapshot.json"]:
    if not (ML_ROOT / name).is_file():
        raise FileNotFoundError(str(ML_ROOT / name))


## 2) データ準備関数

保存済みのクラスタ所属と標準化スケールを再利用します。第3回では61地域それぞれについて他の60地域との距離を計算し、近い5地域を保存します。クラスタの再学習はしません。

In [ ]:
"""第2回の保存データから第3回の参照用レコードを作る。外部通信・Spark書込はしない。"""
import csv
import hashlib
import json
import math
from collections import Counter
from pathlib import Path

FEATURES = ["population_growth_pct", "single_share_change_pp", "elderly_share_change_pp"]
FLOAT_COLUMNS = [
    "single_households_pct_2020", "age_65plus_share_known_pct_2020", "age_unknown_pct_2020",
    "single_households_pct_2025", "age_65plus_share_known_pct_2025", "age_unknown_pct_2025",
    *FEATURES, "age_unknown_change_pp",
]
INT_COLUMNS = ["area_code", "population_2020", "population_2025"]


def require(condition, message):
    if not condition:
        raise ValueError(message)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def prepare_records(csv_paths, ml_root, expected_comparison="compare61_20261020_01",
                    expected_ai="change_ml_20261020_01"):
    ml_root = Path(ml_root)
    metadata = load_json(ml_root / "run_metadata.json")
    params = load_json(ml_root / "model_parameters.json")
    payload = load_json(ml_root / "profile_payload.json")
    validation = load_json(ml_root / "comparison_validation_snapshot.json")
    require(metadata["comparison_run_id"] == expected_comparison, "比較Run IDが一致しません")
    require(metadata["ai_run_id"] == expected_ai, "AI Run IDが一致しません")
    require(payload["ai_run_id"] == expected_ai, "profile_payloadのAI Run IDが不一致")
    require(payload["comparison_run_id"] == expected_comparison, "profile_payloadの比較Run IDが不一致")
    require(validation["comparison_run_id"] == expected_comparison, "validationの比較Run IDが不一致")
    require(validation["status"] == "PASS" and validation["accepted"] is True, "比較条件が未承認です")
    require(params["features"] == FEATURES, "特徴量の順序が異なります")
    require(params == metadata["model_parameters"], "モデルパラメータがmetadataと不一致です")
    require(validation["period"] == "2020-2025", "想定外の比較期間です")
    require(set(map(str, validation["excluded_area_codes"])) == {"13209"}, "対象地域の条件が異なります")
    require(payload["municipality_count"] == 61 and validation["rows"] == 61, "想定する件数は61です")
    scales = params["scaler_scale"]
    require(len(scales) == 3 and all(math.isfinite(s) and s > 0 for s in scales), "標準化のスケールが不正です")

    membership = {}
    for cluster in payload["clusters"]:
        require(cluster["member_count"] == len(cluster["members"]), "クラスタ内件数が不一致")
        for member in cluster["members"]:
            code = int(member["area_code"])
            require(code not in membership, "クラスタ所属に地域コードの重複があります")
            membership[code] = (member["area_name"], int(cluster["cluster_id"]))

    require(csv_paths, "CSVが見つかりません。csv/tokyo_changes内のpart-*.csvを確認してください")
    rows, file_checksums = [], {}
    for path in sorted(map(Path, csv_paths)):
        file_checksums[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
        with path.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle)
            needed = set(INT_COLUMNS + FLOAT_COLUMNS + ["area_name"])
            require(needed <= set(reader.fieldnames or []), f"列が不足しています: {path}")
            for raw in reader:
                row = {k: int(raw[k]) for k in INT_COLUMNS}
                row["area_name"] = raw["area_name"]
                for key in FLOAT_COLUMNS:
                    value = raw[key].strip()
                    row[key] = None if value in ("", "NULL", "null") else float(value)
                    require(row[key] is None or math.isfinite(row[key]), f"非有限値: {key}")
                require(row["area_code"] in membership, "CSVとクラスタ所属の地域集合が異なります")
                name, cluster_id = membership[row["area_code"]]
                require(name == row["area_name"], "地域名が一致しません")
                row["cluster_id"] = cluster_id
                require(all(row[f] is not None for f in FEATURES), "特徴量にNULLがあります")
                require(row["population_2020"] > 0, "基準人口は正数である必要があります")
                expected = {
                    "population_growth_pct": (row["population_2025"] / row["population_2020"] - 1) * 100,
                    "single_share_change_pp": row["single_households_pct_2025"] - row["single_households_pct_2020"],
                    "elderly_share_change_pp": row["age_65plus_share_known_pct_2025"] - row["age_65plus_share_known_pct_2020"],
                }
                a, b = row["age_unknown_pct_2020"], row["age_unknown_pct_2025"]
                expected["age_unknown_change_pp"] = None if a is None or b is None else b - a
                for key, value in expected.items():
                    if value is None:
                        require(row[key] is None, "NULLがゼロ等に置き換わっています")
                    else:
                        require(row[key] is not None and math.isclose(row[key], value, abs_tol=1e-8), f"計算不一致: {key}")
                row.update(comparison_run_id=expected_comparison, ai_run_id=expected_ai)
                rows.append(row)
    rows.sort(key=lambda r: r["area_code"])
    require(len(rows) == 61, f"件数不一致: {len(rows)}")
    require(len({r["area_code"] for r in rows}) == 61, "地域コードが重複しています")
    require(set(r["area_code"] for r in rows) == set(membership), "CSVとクラスタ所属が不一致")
    require(13209 not in membership, "町田市が含まれています")
    counts = Counter(r["cluster_id"] for r in rows)
    require(dict(counts) == {0: 17, 1: 44}, f"クラスタ構成が異なります: {counts}")
    # 保存済みクラスタ中心とCSVの平均も照合。Run IDだけで数値一致とは判断しない。
    for cid in counts:
        subset = [r for r in rows if r["cluster_id"] == cid]
        for i, feature in enumerate(FEATURES):
            avg = sum(r[feature] for r in subset) / len(subset)
            require(math.isclose(avg, params["cluster_centers_original_units"][cid][i], abs_tol=1e-8),
                    f"CSVと保存済みクラスタ中心の不一致: cluster={cid}, feature={feature}")

    neighbors = []
    for src in rows:
        candidates = []
        for dst in rows:
            if src["area_code"] == dst["area_code"]:
                continue
            distance = math.sqrt(sum(((src[f] - dst[f]) / scale) ** 2 for f, scale in zip(FEATURES, scales)))
            candidates.append((distance, dst["area_code"], dst))
        for rank, (distance, _, dst) in enumerate(sorted(candidates)[:5], 1):
            neighbors.append({
                "source_area_code": src["area_code"], "source_area_name": src["area_name"],
                "neighbor_rank": rank, "neighbor_area_code": dst["area_code"], "neighbor_area_name": dst["area_name"],
                "distance_standardized": distance, "neighbor_cluster_id": dst["cluster_id"],
                "population_growth_pct": dst["population_growth_pct"],
                "single_share_change_pp": dst["single_share_change_pp"],
                "elderly_share_change_pp": dst["elderly_share_change_pp"],
                "age_unknown_pct_2020": dst["age_unknown_pct_2020"],
                "age_unknown_pct_2025": dst["age_unknown_pct_2025"],
                "age_unknown_change_pp": dst["age_unknown_change_pp"],
                "comparison_run_id": expected_comparison, "ai_run_id": expected_ai,
            })
    require(len(neighbors) == 305, "類似地域件数が不一致です")
    manifest = {
        "comparison_run_id": expected_comparison, "ai_run_id": expected_ai,
        "area_rows": len(rows), "neighbor_rows": len(neighbors), "cluster_counts": dict(counts),
        "source_csv_file_sha256": file_checksums,
        "similarity": "Euclidean distance on the three features using saved scaler_scale; all 60 other municipalities are candidates",
        "note": "CSV bytes hash is a new file hash, not the row-level/Parquet hash recorded in Part 2",
    }
    return rows, neighbors, manifest


## 3) 61件の地域データと305件の類似地域データを検証する

In [ ]:
area_rows, neighbor_rows, manifest = prepare_records(
    CSV_PATHS, ML_ROOT, COMPARISON_RUN_ID, AI_RUN_ID
)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print(json.dumps(next(r for r in area_rows if r["area_code"] == 13201), ensure_ascii=False, indent=2))


## 4) 新規のDeltaテーブルを2つ作成する

既存テーブルがあると停止します。再実行時は既存テーブルを確認したうえで、今回の2テーブルを再作成するときだけ `ALLOW_REPLACE_PART3_TABLES=True` に変更してください。新しいsuffixを使う場合はSQL Tool側のテーブル名も揃えます。

In [ ]:
import re
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType

for identifier in (CATALOG, SCHEMA, TABLE_SUFFIX):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError("保存先の識別子が不正です")
AREA_TABLE = f"{CATALOG}.{SCHEMA}.census_agent_areas_{TABLE_SUFFIX}"
NEIGHBOR_TABLE = f"{CATALOG}.{SCHEMA}.census_agent_neighbors_{TABLE_SUFFIX}"
targets = [(AREA_TABLE, area_rows), (NEIGHBOR_TABLE, neighbor_rows)]
existing = [name for name, _ in targets if spark.catalog.tableExists(name)]
if existing and not ALLOW_REPLACE_PART3_TABLES:
    raise RuntimeError(f"既存テーブルがあります。内容を確認してから再実行してください: {existing}")

def to_spark_frame(records):
    keys = list(records[0])
    fields = []
    for key in keys:
        sample = next((r[key] for r in records if r[key] is not None), None)
        if sample is None:
            raise ValueError(f"型を決定できません: {key}")
        kind = LongType() if isinstance(sample, int) else DoubleType() if isinstance(sample, float) else StringType()
        fields.append(StructField(key, kind, True))
    return spark.createDataFrame([tuple(row[k] for k in keys) for row in records], StructType(fields))

for name, records in targets:
    frame = to_spark_frame(records)
    writer = frame.write.format("delta")
    if ALLOW_REPLACE_PART3_TABLES:
        writer = writer.mode("overwrite").option("overwriteSchema", "true")
    else:
        writer = writer.mode("errorifexists")
    writer.saveAsTable(name)
    actual = spark.table(name).count()
    require(actual == len(records), f"書込後の件数不一致: {name}")
    print(name, actual)


## 5) Spark SQLでToolと同じクエリを確認する

この確認はNotebookに接続中のSpark Compute上で実行します。続いてAgentのSQL Toolに同じComputeを選択します。

In [ ]:
QUERIES = {'01_list_areas': 'SELECT area_code, area_name, cluster_id\nFROM blog_catalog.opendata.census_agent_areas_v1\nORDER BY area_code', '02_get_area_metrics': 'SELECT area_code, area_name, cluster_id,\n       population_2020, population_2025,\n       ROUND(population_growth_pct, 2) AS population_growth_pct,\n       ROUND(single_households_pct_2020, 2) AS single_households_pct_2020,\n       ROUND(single_households_pct_2025, 2) AS single_households_pct_2025,\n       ROUND(single_share_change_pp, 2) AS single_share_change_pp,\n       ROUND(age_65plus_share_known_pct_2020, 2) AS age_65plus_share_known_pct_2020,\n       ROUND(age_65plus_share_known_pct_2025, 2) AS age_65plus_share_known_pct_2025,\n       ROUND(elderly_share_change_pp, 2) AS elderly_share_change_pp,\n       ROUND(age_unknown_pct_2020, 2) AS age_unknown_pct_2020,\n       ROUND(age_unknown_pct_2025, 2) AS age_unknown_pct_2025,\n       ROUND(age_unknown_change_pp, 2) AS age_unknown_change_pp,\n       comparison_run_id, ai_run_id\nFROM blog_catalog.opendata.census_agent_areas_v1\nWHERE area_code = {{area_code}}', '03_rank_population_growth': 'SELECT area_code, area_name, cluster_id,\n       population_2020, population_2025,\n       ROUND(population_growth_pct, 2) AS population_growth_pct,\n       comparison_run_id\nFROM blog_catalog.opendata.census_agent_areas_v1 AS a\nORDER BY a.population_growth_pct DESC, a.area_code\nLIMIT 5', '04_find_similar_areas': 'SELECT source_area_code, source_area_name, neighbor_rank,\n       neighbor_area_code, neighbor_area_name, neighbor_cluster_id,\n       ROUND(distance_standardized, 6) AS distance_standardized,\n       ROUND(population_growth_pct, 2) AS population_growth_pct,\n       ROUND(single_share_change_pp, 2) AS single_share_change_pp,\n       ROUND(elderly_share_change_pp, 2) AS elderly_share_change_pp,\n       ROUND(age_unknown_pct_2020, 2) AS age_unknown_pct_2020,\n       ROUND(age_unknown_pct_2025, 2) AS age_unknown_pct_2025,\n       ROUND(age_unknown_change_pp, 2) AS age_unknown_change_pp,\n       comparison_run_id, ai_run_id\nFROM blog_catalog.opendata.census_agent_neighbors_v1\nWHERE source_area_code = {{area_code}}\nORDER BY neighbor_rank', '05_get_cluster_summary': 'SELECT cluster_id, COUNT(*) AS member_count,\n       ROUND(AVG(population_growth_pct), 2) AS mean_population_growth_pct,\n       ROUND(AVG(single_share_change_pp), 2) AS mean_single_share_change_pp,\n       ROUND(AVG(elderly_share_change_pp), 2) AS mean_elderly_share_change_pp,\n       COUNT(age_unknown_change_pp) AS age_unknown_change_available_count,\n       COUNT(*) - COUNT(age_unknown_change_pp) AS age_unknown_change_null_count,\n       ROUND(AVG(age_unknown_change_pp), 2) AS mean_age_unknown_change_pp,\n       ROUND(MAX(ABS(age_unknown_change_pp)), 2) AS max_abs_age_unknown_change_pp,\n       comparison_run_id, ai_run_id\nFROM blog_catalog.opendata.census_agent_areas_v1\nGROUP BY cluster_id, comparison_run_id, ai_run_id\nORDER BY cluster_id'}

for name, query in QUERIES.items():
    query = query.replace("blog_catalog.opendata.census_agent_areas_v1", AREA_TABLE)
    query = query.replace("blog_catalog.opendata.census_agent_neighbors_v1", NEIGHBOR_TABLE)
    query = query.replace("{{area_code}}", "13201")
    print(name)
    spark.sql(query).show(61 if name == "01_list_areas" else 5, truncate=False)


## 完了条件

- 地域テーブル61件、類似地域テーブル305件。
- クラスタ0は17件、クラスタ1は44件。
- 八王子市の年齢不詳率差は11.39pp。
- 八王子市の類似地域は羽村市、渋谷区、日の出町、青梅市、多摩市の順。
- Agentの実行前にSpark ComputeがActiveであることを確認。検証終了時はAI ComputeとSpark Computeを停止。
